# 04 — Auditoria de Integridade Física, Referencial, Privacidade e Prontidão do Dataset (Eleições 2026)

Este notebook autocontido implementa a **Fase 3 (`phase-3-dataset-integrity-and-release`)** do pipeline analítico 1:1 das Eleições Gerais de 2026 do TSE:

1. **Contrato Canônico 1:1 das 18 Tabelas (`fact_*`, `dim_*`, `bridge_*`)**:
   - Verifica os schemas nativos exatos de 2026 (`AA_ELEICAO` em Eleitorado/Correspondências vs `ANO_ELEICAO` em BU/Resultados/Comparecimento; `CD_GRAU_ESCOLARIDADE` vs `CD_GRAU_INSTRUCAO`; 15 colunas de `fact_urna_logs`).
2. **Auditoria de Partições Hive e Reconciliação com `_checkpoints/manifest.json`**:
   - Valida a presença física, compressão `ZSTD`, paridade de `row_count`, verificação `.sha512` (`fact_boletim_urna`, `bridge_correspondencia_esperada`, `bridge_correspondencia_efetivada`) e ausência de chaves de partição proibidas (`section_num`, `urna_serial`, `municipio_code`, `raw_timestamp`).
3. **Integridade Referencial e Cruzamento de Chaves via DuckDB**:
   - Reconciliação de seções, votos (`QT_VOTOS`) e comparecimento (`QT_APTOS`, `QT_COMPARECIMENTO`, `QT_ABSTENCOES`) entre `fact_boletim_urna`, `fact_votacao_secao` e `fact_detalhe_votacao_secao`.
   - Cruzamento de `urna_serial` (`fact_urna_logs`) com `NR_URNA_EFETIVADA` e `NR_URNA_ESPERADA`, reportando cobertura explícita por UF e turno e tratando seções sem `.jez` como `missing_log` (`NULL`), nunca como zero ocorrências.
   - Consistência contábil e quantitativa de eleitores aptos entre `fact_eleitorado_perfil_secao`, `dim_local_votacao_secao` e `fact_comparecimento_abstencao` (`QT_APTOS == QT_COMPARECIMENTO + QT_ABSTENCAO`).
4. **Auditoria Final de Privacidade, Orçamento de Volume ZSTD e Geração de Evidências (`integrity_audit_report.json` e `dataset-metadata.json`)**:
   - Confirma ausência física de `mac_code` e zero sequências numéricas brutas de 5+ dígitos sem `<ID_REDACTED>` em `fact_urna_logs.message_text`.
   - Calcula o volume ZSTD por tabela e gera as evidências para revisão humana antes da publicação manual no Kaggle.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import tempfile
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import duckdb
import pyarrow.parquet as pq


class PrivacyAuditError(RuntimeError):
    """Exceção lançada quando a auditoria de privacidade detecta mac_code ou 5+ dígitos desmascarados."""


FORBIDDEN_PARTITION_KEYS: set[str] = {
    "section_num",
    "urna_serial",
    "municipio_code",
    "raw_timestamp",
}
ALLOWED_PARTITION_KEYS: set[str] = {"election_year", "turn", "uf"}
EXPECTED_ELECTORAL_UFS: set[str] = {
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA", "MG", "MS",
    "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN", "RO", "RR", "RS", "SC",
    "SE", "SP", "TO", "ZZ",
}

VERIFIED_TSE_2026_URLS: list[str] = [
    "https://www.tse.jus.br/eleicoes/eleicoes-2026-content/arquivos/log-do-ecossistema/@@display-file/file/log-do-ecossistema.pdf",
    "https://www.tse.jus.br/legislacao/compilada/res/2026/resolucao-no-23-759-de-26-de-fevereiro-de-2026",
    "https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes2026/arqurnatot/bu_imgbu_logjez_rdv_vscmr_2026_{turn}t_{uf}.zip",
    "https://dadosabertos.tse.jus.br/api/3/action/package_show?id=resultados-2026-boletim-de-urna",
    "https://dadosabertos.tse.jus.br/api/3/action/package_show?id=resultados-2026",
    "https://dadosabertos.tse.jus.br/api/3/action/package_show?id=eleitorado-2026",
    "https://dadosabertos.tse.jus.br/api/3/action/package_show?id=comparecimento-e-abstencao-2026",
    "https://dadosabertos.tse.jus.br/api/3/action/package_show?id=resultados-2026-correspondencias-esperadas-e-efetivadas-1-turno",
    "https://dadosabertos.tse.jus.br/api/3/action/package_show?id=resultados-2026-correspondencias-esperadas-e-efetivadas-2-turno",
]

TABLE_CATALOG_18: dict[str, dict[str, Any]] = {
    "fact_urna_logs": {
        "phase": 2,
        "column_count": 15,
        "year_column": "election_year",
        "requires_sha512": False,
        "description": "Eventos cronológicos brutos extraídos de logd.dat (.jez) da urna eletrônica com anonimização <ID_REDACTED> e omissão de mac_code.",
        "required_columns": [
            "election_year",
            "turn",
            "uf",
            "cd_pleito",
            "cd_municipio",
            "nr_zona",
            "nr_secao",
            "urna_serial",
            "message_text",
        ],
        "expected_columns": [
            "election_year",
            "turn",
            "uf",
            "cd_pleito",
            "cd_municipio",
            "nr_zona",
            "nr_secao",
            "source_jez_file",
            "source_log_member",
            "file_seq",
            "raw_timestamp",
            "log_level",
            "urna_serial",
            "application",
            "message_text",
        ],
    },
    "fact_boletim_urna": {
        "phase": 1,
        "column_count": 45,
        "year_column": "ANO_ELEICAO",
        "requires_sha512": True,
        "description": "Boletim de Urna (BU) por seção eleitoral, cargo e votável (buweb/bweb_{turn}t_{uf}_<DDMMYYYYHHMM>.zip).",
        "required_columns": [
            "ANO_ELEICAO",
            "NR_TURNO",
            "SG_UF",
            "CD_PLEITO",
            "CD_MUNICIPIO",
            "NR_ZONA",
            "NR_SECAO",
            "CD_CARGO_PERGUNTA",
            "NR_VOTAVEL",
            "QT_VOTOS",
            "NR_URNA_EFETIVADA",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "ANO_ELEICAO",
            "CD_TIPO_ELEICAO",
            "NM_TIPO_ELEICAO",
            "CD_PLEITO",
            "DT_PLEITO",
            "NR_TURNO",
            "CD_ELEICAO",
            "DS_ELEICAO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NM_MUNICIPIO",
            "NR_ZONA",
            "NR_SECAO",
            "NR_LOCAL_VOTACAO",
            "CD_CARGO_PERGUNTA",
            "DS_CARGO_PERGUNTA",
            "NR_PARTIDO",
            "SG_PARTIDO",
            "NM_PARTIDO",
            "DT_BU_RECEBIDO",
            "QT_APTOS",
            "QT_COMPARECIMENTO",
            "QT_ABSTENCOES",
            "CD_TIPO_URNA",
            "DS_TIPO_URNA",
            "CD_TIPO_VOTAVEL",
            "DS_TIPO_VOTAVEL",
            "NR_VOTAVEL",
            "NM_VOTAVEL",
            "QT_VOTOS",
            "NR_URNA_EFETIVADA",
            "CD_CARGA_1_URNA_EFETIVADA",
            "CD_CARGA_2_URNA_EFETIVADA",
            "CD_FLASHCARD_URNA_EFETIVADA",
            "DT_CARGA_URNA_EFETIVADA",
            "DS_CARGO_PERGUNTA_SECAO",
            "DS_SECOES_AGREGADAS",
            "DT_ABERTURA",
            "DT_ENCERRAMENTO",
            "QT_ELEI_BIOM_SEM_HABILITACAO",
            "DT_EMISSAO_BU",
            "NR_JUNTA_APURADORA",
            "NR_TURMA_APURADORA",
        ],
    },
    "fact_votacao_secao": {
        "phase": 1,
        "column_count": 26,
        "year_column": "ANO_ELEICAO",
        "requires_sha512": False,
        "description": "Votação nominal/legenda/branco/nulo apurada por seção eleitoral e cargo (votacao_secao_2026_{uf}.zip).",
        "required_columns": [
            "ANO_ELEICAO",
            "NR_TURNO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NR_ZONA",
            "NR_SECAO",
            "CD_CARGO",
            "NR_VOTAVEL",
            "QT_VOTOS",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "ANO_ELEICAO",
            "CD_TIPO_ELEICAO",
            "NM_TIPO_ELEICAO",
            "NR_TURNO",
            "CD_ELEICAO",
            "DS_ELEICAO",
            "DT_ELEICAO",
            "TP_ABRANGENCIA",
            "SG_UF",
            "SG_UE",
            "NM_UE",
            "CD_MUNICIPIO",
            "NM_MUNICIPIO",
            "NR_ZONA",
            "NR_SECAO",
            "CD_CARGO",
            "DS_CARGO",
            "NR_VOTAVEL",
            "NM_VOTAVEL",
            "QT_VOTOS",
            "NR_LOCAL_VOTACAO",
            "SQ_CANDIDATO",
            "NM_LOCAL_VOTACAO",
            "DS_LOCAL_VOTACAO_ENDERECO",
        ],
    },
    "fact_detalhe_votacao_secao": {
        "phase": 1,
        "column_count": 37,
        "year_column": "ANO_ELEICAO",
        "requires_sha512": False,
        "description": "Detalhe de apuração, comparecimento e abstenção por seção e cargo (detalhe_votacao_secao_2026.zip).",
        "required_columns": [
            "ANO_ELEICAO",
            "NR_TURNO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NR_ZONA",
            "NR_SECAO",
            "CD_CARGO",
            "QT_APTOS",
            "QT_COMPARECIMENTO",
            "QT_ABSTENCOES",
            "ST_SECAO_ANULADA",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "ANO_ELEICAO",
            "CD_TIPO_ELEICAO",
            "NM_TIPO_ELEICAO",
            "NR_TURNO",
            "CD_ELEICAO",
            "DS_ELEICAO",
            "DT_ELEICAO",
            "TP_ABRANGENCIA",
            "SG_UF",
            "SG_UE",
            "NM_UE",
            "CD_MUNICIPIO",
            "NM_MUNICIPIO",
            "NR_ZONA",
            "NR_SECAO",
            "CD_CARGO",
            "DS_CARGO",
            "QT_APTOS",
            "QT_COMPARECIMENTO",
            "QT_ABSTENCOES",
            "QT_VOTOS_NOMINAIS",
            "QT_VOTOS_BRANCOS",
            "QT_VOTOS_NULOS",
            "QT_VOTOS_LEGENDA",
            "QT_VOTOS_ANULADOS_APU_SEP",
            "NR_LOCAL_VOTACAO",
            "NM_LOCAL_VOTACAO",
            "DS_LOCAL_VOTACAO_ENDERECO",
            "DT_RECEBIMENTO_BU_HOR_TSE",
            "DT_PRIM_TOT_PARCIAL_HOR_TSE",
            "DS_ORIGEM_VOTO",
            "ST_SECAO_INSTALADA",
            "ST_SECAO_ANULADA",
            "CD_MODELO_URNA",
            "DS_MODELO_URNA",
        ],
    },
    "fact_detalhe_votacao_munzona": {
        "phase": 1,
        "column_count": 47,
        "year_column": "ANO_ELEICAO",
        "requires_sha512": False,
        "description": "Consolidado de apuração e comparecimento por município, zona e cargo (detalhe_votacao_munzona_2026.zip).",
        "required_columns": [
            "ANO_ELEICAO",
            "NR_TURNO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NR_ZONA",
            "CD_CARGO",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "ANO_ELEICAO",
            "CD_TIPO_ELEICAO",
            "NM_TIPO_ELEICAO",
            "NR_TURNO",
            "CD_ELEICAO",
            "DS_ELEICAO",
            "DT_ELEICAO",
            "TP_ABRANGENCIA",
            "SG_UF",
            "SG_UE",
            "NM_UE",
            "CD_MUNICIPIO",
            "NM_MUNICIPIO",
            "NR_ZONA",
            "CD_CARGO",
            "DS_CARGO",
            "QT_APTOS",
            "QT_SECOES_PRINCIPAIS",
            "QT_SECOES_AGREGADAS",
            "QT_SECOES_NAO_INSTALADAS",
            "QT_TOTAL_SECOES",
            "QT_COMPARECIMENTO",
            "QT_ELEITORES_SECOES_NAO_INSTALADAS",
            "QT_ABSTENCOES",
            "ST_VOTO_EM_TRANSITO",
            "QT_VOTOS",
            "QT_VOTOS_CONCORRENTES",
            "QT_TOTAL_VOTOS_VALIDOS",
            "QT_VOTOS_NOMINAIS_VALIDOS",
            "QT_TOTAL_VOTOS_LEG_VALIDOS",
            "QT_VOTOS_LEG_VALIDOS",
            "QT_VOTOS_NOM_CONVR_LEG_VALIDOS",
            "QT_TOTAL_VOTOS_ANULADOS",
            "QT_VOTOS_NOMINAIS_ANULADOS",
            "QT_VOTOS_LEGENDA_ANULADOS",
            "QT_TOTAL_VOTOS_ANUL_SUBJUD",
            "QT_VOTOS_NOMINAIS_ANUL_SUBJUD",
            "QT_VOTOS_LEGENDA_ANUL_SUBJUD",
            "QT_VOTOS_BRANCOS",
            "QT_TOTAL_VOTOS_NULOS",
            "QT_VOTOS_NULOS",
            "QT_VOTOS_NULOS_TECNICOS",
            "QT_VOTOS_ANULADOS_APU_SEP",
            "HH_ULTIMA_TOTALIZACAO",
            "DT_ULTIMA_TOTALIZACAO",
        ],
    },
    "dim_candidato_munzona": {
        "phase": 1,
        "column_count": 50,
        "year_column": "ANO_ELEICAO",
        "requires_sha512": False,
        "description": "Dimensão e votação de candidatos por município e zona (votacao_candidato_munzona_2026.zip).",
        "required_columns": [
            "ANO_ELEICAO",
            "NR_TURNO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NR_ZONA",
            "CD_CARGO",
            "SQ_CANDIDATO",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "ANO_ELEICAO",
            "CD_TIPO_ELEICAO",
            "NM_TIPO_ELEICAO",
            "NR_TURNO",
            "CD_ELEICAO",
            "DS_ELEICAO",
            "DT_ELEICAO",
            "TP_ABRANGENCIA",
            "SG_UF",
            "SG_UE",
            "NM_UE",
            "CD_MUNICIPIO",
            "NM_MUNICIPIO",
            "NR_ZONA",
            "CD_CARGO",
            "DS_CARGO",
            "SQ_CANDIDATO",
            "NR_CANDIDATO",
            "NM_CANDIDATO",
            "NM_URNA_CANDIDATO",
            "NM_SOCIAL_CANDIDATO",
            "CD_SITUACAO_CANDIDATURA",
            "DS_SITUACAO_CANDIDATURA",
            "CD_DETALHE_SITUACAO_CAND",
            "DS_DETALHE_SITUACAO_CAND",
            "CD_SITUACAO_JULGAMENTO",
            "DS_SITUACAO_JULGAMENTO",
            "CD_SITUACAO_CASSACAO",
            "DS_SITUACAO_CASSACAO",
            "CD_SITUACAO_DCONST_DIPLOMA",
            "DS_SITUACAO_DCONST_DIPLOMA",
            "TP_AGREMIACAO",
            "NR_PARTIDO",
            "SG_PARTIDO",
            "NM_PARTIDO",
            "NR_FEDERACAO",
            "NM_FEDERACAO",
            "SG_FEDERACAO",
            "DS_COMPOSICAO_FEDERACAO",
            "SQ_COLIGACAO",
            "NM_COLIGACAO",
            "DS_COMPOSICAO_COLIGACAO",
            "ST_VOTO_EM_TRANSITO",
            "QT_VOTOS_NOMINAIS",
            "NM_TIPO_DESTINACAO_VOTOS",
            "QT_VOTOS_NOMINAIS_VALIDOS",
            "CD_SIT_TOT_TURNO",
            "DS_SIT_TOT_TURNO",
        ],
    },
    "dim_partido_munzona": {
        "phase": 1,
        "column_count": 38,
        "year_column": "ANO_ELEICAO",
        "requires_sha512": False,
        "requires_source_coverage": True,
        "description": "Dimensão e votação de partidos/federações por município e zona (votacao_partido_munzona_2026.zip).",
        "required_columns": [
            "ANO_ELEICAO",
            "NR_TURNO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NR_ZONA",
            "CD_CARGO",
            "NR_PARTIDO",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "ANO_ELEICAO",
            "CD_TIPO_ELEICAO",
            "NM_TIPO_ELEICAO",
            "NR_TURNO",
            "CD_ELEICAO",
            "DS_ELEICAO",
            "DT_ELEICAO",
            "TP_ABRANGENCIA",
            "SG_UF",
            "SG_UE",
            "NM_UE",
            "CD_MUNICIPIO",
            "NM_MUNICIPIO",
            "NR_ZONA",
            "CD_CARGO",
            "DS_CARGO",
            "TP_AGREMIACAO",
            "NR_PARTIDO",
            "SG_PARTIDO",
            "NM_PARTIDO",
            "NR_FEDERACAO",
            "NM_FEDERACAO",
            "SG_FEDERACAO",
            "DS_COMPOSICAO_FEDERACAO",
            "SQ_COLIGACAO",
            "NM_COLIGACAO",
            "DS_COMPOSICAO_COLIGACAO",
            "ST_VOTO_EM_TRANSITO",
            "QT_VOTOS_LEGENDA_VALIDOS",
            "QT_VOTOS_NOM_CONVR_LEG_VALIDOS",
            "QT_TOTAL_VOTOS_LEG_VALIDOS",
            "QT_VOTOS_NOMINAIS_VALIDOS",
            "QT_VOTOS_LEGENDA_ANUL_SUBJUD",
            "QT_VOTOS_NOMINAIS_ANUL_SUBJUD",
            "QT_VOTOS_LEGENDA_ANULADOS",
            "QT_VOTOS_NOMINAIS_ANULADOS",
        ],
    },
    "fact_eleitorado_perfil_secao": {
        "phase": 1,
        "column_count": 30,
        "year_column": "AA_ELEICAO",
        "requires_sha512": False,
        "description": "Perfil demográfico e biométrico do eleitorado por seção eleitoral (perfil_eleitor_secao_2026_{uf}.zip).",
        "required_columns": [
            "AA_ELEICAO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NR_ZONA",
            "NR_SECAO",
            "CD_GENERO",
            "CD_FAIXA_ETARIA",
            "CD_GRAU_ESCOLARIDADE",
            "QT_ELEITORES",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "AA_ELEICAO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NM_MUNICIPIO",
            "NR_ZONA",
            "NR_SECAO",
            "NR_LOCAL_VOTACAO",
            "NM_LOCAL_VOTACAO",
            "CD_GENERO",
            "DS_GENERO",
            "CD_ESTADO_CIVIL",
            "DS_ESTADO_CIVIL",
            "CD_FAIXA_ETARIA",
            "DS_FAIXA_ETARIA",
            "CD_GRAU_ESCOLARIDADE",
            "DS_GRAU_ESCOLARIDADE",
            "CD_RACA_COR",
            "DS_RACA_COR",
            "CD_IDENTIDADE_GENERO",
            "DS_IDENTIDADE_GENERO",
            "CD_QUILOMBOLA",
            "DS_QUILOMBOLA",
            "CD_INTERPRETE_LIBRAS",
            "DS_INTERPRETE_LIBRAS",
            "QT_ELEITORES",
            "QT_ELEITORES_BIOMETRIA",
            "QT_ELEITORES_DEFICIENCIA",
            "QT_ELEITORES_NOME_SOCIAL",
        ],
    },
    "fact_eleitorado_perfil_munzona": {
        "phase": 1,
        "column_count": 27,
        "year_column": "AA_ELEICAO",
        "requires_sha512": False,
        "description": "Perfil demográfico do eleitorado por município e zona (perfil_eleitorado_2026.zip).",
        "required_columns": [
            "AA_ELEICAO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NR_ZONA",
            "CD_GENERO",
            "CD_FAIXA_ETARIA",
            "CD_GRAU_ESCOLARIDADE",
            "QT_ELEITORES",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "AA_ELEICAO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NM_MUNICIPIO",
            "NR_ZONA",
            "CD_GENERO",
            "DS_GENERO",
            "CD_ESTADO_CIVIL",
            "DS_ESTADO_CIVIL",
            "CD_FAIXA_ETARIA",
            "DS_FAIXA_ETARIA",
            "CD_GRAU_ESCOLARIDADE",
            "DS_GRAU_ESCOLARIDADE",
            "CD_RACA_COR",
            "DS_RACA_COR",
            "CD_IDENTIDADE_GENERO",
            "DS_IDENTIDADE_GENERO",
            "CD_QUILOMBOLA",
            "DS_QUILOMBOLA",
            "CD_INTERPRETE_LIBRAS",
            "DS_INTERPRETE_LIBRAS",
            "QT_ELEITORES",
            "QT_ELEITORES_BIOMETRIA",
            "QT_ELEITORES_DEFICIENCIA",
            "QT_ELEITORES_NOME_SOCIAL",
        ],
    },
    "dim_local_votacao_secao": {
        "phase": 1,
        "column_count": 41,
        "year_column": "AA_ELEICAO",
        "requires_sha512": False,
        "description": "Cadastro geolocalizado de locais de votação e seções agregadas/principais (eleitorado_local_votacao_2026.zip).",
        "required_columns": [
            "AA_ELEICAO",
            "NR_TURNO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NR_ZONA",
            "NR_SECAO",
            "NR_LOCAL_VOTACAO",
            "NR_SECAO_PRINCIPAL",
            "QT_ELEITOR_SECAO",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "AA_ELEICAO",
            "DT_ELEICAO",
            "DS_ELEICAO",
            "NR_TURNO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NM_MUNICIPIO",
            "NR_ZONA",
            "NR_SECAO",
            "CD_TIPO_SECAO_AGREGADA",
            "DS_TIPO_SECAO_AGREGADA",
            "NR_SECAO_PRINCIPAL",
            "NR_LOCAL_VOTACAO",
            "NM_LOCAL_VOTACAO",
            "CD_TIPO_LOCAL",
            "DS_TIPO_LOCAL",
            "DS_ENDERECO",
            "NM_BAIRRO",
            "NR_CEP",
            "NR_TELEFONE_LOCAL",
            "NR_LATITUDE",
            "NR_LONGITUDE",
            "CD_SITU_LOCAL_VOTACAO",
            "DS_SITU_LOCAL_VOTACAO",
            "CD_SITU_ZONA",
            "DS_SITU_ZONA",
            "CD_SITU_SECAO",
            "DS_SITU_SECAO",
            "CD_SITU_LOCALIDADE",
            "DS_SITU_LOCALIDADE",
            "CD_SITU_SECAO_ACESSIBILIDADE",
            "DS_SITU_SECAO_ACESSIBILIDADE",
            "QT_ELEITOR_SECAO",
            "QT_ELEITOR_ELEICAO_FEDERAL",
            "QT_ELEITOR_ELEICAO_ESTADUAL",
            "QT_ELEITOR_ELEICAO_MUNICIPAL",
            "NR_LOCAL_VOTACAO_ORIGINAL",
            "NM_LOCAL_VOTACAO_ORIGINAL",
            "DS_ENDERECO_LOCVT_ORIGINAL",
        ],
    },
    "fact_eleitorado_deficiencia": {
        "phase": 1,
        "column_count": 30,
        "year_column": "AA_ELEICAO",
        "requires_sha512": False,
        "description": "Perfil de eleitores com deficiência por seção eleitoral (perfil_eleitor_deficiencia_2026.zip).",
        "required_columns": [
            "AA_ELEICAO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NR_ZONA",
            "NR_SECAO",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "AA_ELEICAO",
            "SQ_ELEITOR",
            "SG_UF",
            "CD_MUNICIPIO",
            "NM_MUNICIPIO",
            "CD_MUN_SITU_BIOMETRICA",
            "DS_MUN_SITU_BIOMETRICA",
            "NR_ZONA",
            "NR_SECAO",
            "CD_GENERO",
            "DS_GENERO",
            "CD_ESTADO_CIVIL",
            "DS_ESTADO_CIVIL",
            "CD_FAIXA_ETARIA",
            "DS_FAIXA_ETARIA",
            "CD_GRAU_ESCOLARIDADE",
            "DS_GRAU_ESCOLARIDADE",
            "CD_TIPO_DEFICIENCIA",
            "DS_TIPO_DEFICIENCIA",
            "ST_ELEITOR_BIOMETRIA",
            "CD_RACA_COR",
            "DS_RACA_COR",
            "CD_IDENTIDADE_GENERO",
            "DS_IDENTIDADE_GENERO",
            "CD_QUILOMBOLA",
            "DS_QUILOMBOLA",
            "CD_INTERPRETE_LIBRAS",
            "DS_INTERPRETE_LIBRAS",
        ],
    },
    "fact_transferencia_temporaria_secao": {
        "phase": 1,
        "column_count": 17,
        "year_column": "AA_ELEICAO",
        "requires_sha512": False,
        "description": "Transferências temporárias de eleitores (TTE) por seção de origem e destino (transf_temporaria_secao_2026.zip).",
        "required_columns": [
            "AA_ELEICAO",
            "NR_TURNO",
            "SG_UF_ORIGEM",
            "CD_MUNICIPIO_ORIGEM",
            "NR_ZONA_ORIGEM",
            "NR_SECAO_ORIGEM",
            "SG_UF_DESTINO",
            "CD_MUNICIPIO_DESTINO",
            "NR_ZONA_DESTINO",
            "NR_SECAO_DESTINO",
            "QT_ELEITOR",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "AA_ELEICAO",
            "NR_TURNO",
            "TP_TTE",
            "TP_ABRANGENCIA_TTE",
            "SG_UF_ORIGEM",
            "CD_MUNICIPIO_ORIGEM",
            "NM_MUNICIPIO_ORIGEM",
            "NR_ZONA_ORIGEM",
            "NR_SECAO_ORIGEM",
            "SG_UF_DESTINO",
            "CD_MUNICIPIO_DESTINO",
            "NM_MUNICIPIO_DESTINO",
            "NR_ZONA_DESTINO",
            "NR_SECAO_DESTINO",
            "QT_ELEITOR",
        ],
    },
    "fact_transferencia_temporaria_mun": {
        "phase": 1,
        "column_count": 13,
        "year_column": "AA_ELEICAO",
        "requires_sha512": False,
        "description": "Transferências temporárias de eleitores (TTE) agregadas por município (transferencia_temporaria_2026.zip).",
        "required_columns": [
            "AA_ELEICAO",
            "NR_TURNO",
            "SG_UF_ORIGEM",
            "CD_MUNICIPIO_ORIGEM",
            "SG_UF_DESTINO",
            "CD_MUNICIPIO_DESTINO",
            "QT_ELEITOR",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "AA_ELEICAO",
            "NR_TURNO",
            "TP_TTE",
            "TP_ABRANGENCIA_TTE",
            "SG_UF_ORIGEM",
            "CD_MUNICIPIO_ORIGEM",
            "NM_MUNICIPIO_ORIGEM",
            "SG_UF_DESTINO",
            "CD_MUNICIPIO_DESTINO",
            "NM_MUNICIPIO_DESTINO",
            "QT_ELEITOR",
        ],
    },
    "fact_comparecimento_abstencao": {
        "phase": 1,
        "column_count": 45,
        "year_column": "ANO_ELEICAO",
        "requires_sha512": False,
        "description": "Perfil demográfico de comparecimento e abstenção por município e zona (Sistema ELO, redistribution_allowed: true).",
        "required_columns": [
            "ANO_ELEICAO",
            "NR_TURNO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NR_ZONA",
            "CD_GENERO",
            "CD_FAIXA_ETARIA",
            "CD_GRAU_INSTRUCAO",
            "QT_APTOS",
            "QT_COMPARECIMENTO",
            "QT_ABSTENCAO",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "ANO_ELEICAO",
            "NR_TURNO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NM_MUNICIPIO",
            "NR_ZONA",
            "CD_GENERO",
            "DS_GENERO",
            "CD_ESTADO_CIVIL",
            "DS_ESTADO_CIVIL",
            "CD_FAIXA_ETARIA",
            "DS_FAIXA_ETARIA",
            "CD_GRAU_INSTRUCAO",
            "DS_GRAU_INSTRUCAO",
            "CD_COR_RACA",
            "DS_COR_RACA",
            "CD_QUILOMBOLA",
            "DS_QUILOMBOLA",
            "CD_INTERPRETE_LIBRAS",
            "DS_INTERPRETE_LIBRAS",
            "CD_IDENTIDADE_GENERO",
            "DS_IDENTIDADE_GENERO",
            "CD_IDIOMA_INDIGENA",
            "DS_IDIOMA_INDIGENA",
            "CD_GRUPO_INDIGENA",
            "DS_GRUPO_INDIGENA",
            "CD_IDIOMA_QUILOMBOLA",
            "DS_IDIOMA_QUILOMBOLA",
            "QT_APTOS",
            "QT_COMPARECIMENTO",
            "QT_ABSTENCAO",
            "QT_COMPARECIMENTO_DEFICIENCIA",
            "QT_ABSTENCAO_DEFICIENCIA",
            "QT_COMPARECIMENTO_TTE",
            "QT_ABSTENCAO_TTE",
            "QT_COMPAREC_FACULTATIVO",
            "QT_ABST_FACULTATIVO",
            "QT_COMPAREC_OBRIGATORIO",
            "QT_ABST_OBRIGATORIO",
            "QT_COMPAREC_DEFIC_FACULTATIVO",
            "QT_ABST_DEFIC_FACULTATIVO",
            "QT_COMPAREC_DEFIC_OBRIGATORIO",
            "QT_ABST_DEFIC_OBRIGATORIO",
        ],
    },
    "fact_comparecimento_abstencao_pcd": {
        "phase": 1,
        "column_count": 20,
        "year_column": "ANO_ELEICAO",
        "requires_sha512": False,
        "description": "Comparecimento e abstenção de eleitores com deficiência (Sistema ELO, redistribution_allowed: true).",
        "required_columns": [
            "ANO_ELEICAO",
            "NR_TURNO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NR_ZONA",
            "CD_GRAU_INSTRUCAO",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "ANO_ELEICAO",
            "NR_TURNO",
            "SQ_ELEITOR",
            "SG_UF",
            "CD_MUNICIPIO",
            "NM_MUNICIPIO",
            "NR_ZONA",
            "CD_GENERO",
            "DS_GENERO",
            "CD_ESTADO_CIVIL",
            "DS_ESTADO_CIVIL",
            "CD_FAIXA_ETARIA",
            "DS_FAIXA_ETARIA",
            "CD_GRAU_INSTRUCAO",
            "DS_GRAU_INSTRUCAO",
            "CD_TIPO_DEFICIENCIA",
            "DS_TIPO_DEFICIENCIA",
            "ST_COMPARECIMENTO",
        ],
    },
    "fact_comparecimento_abstencao_tte": {
        "phase": 1,
        "column_count": 25,
        "year_column": "ANO_ELEICAO",
        "requires_sha512": False,
        "description": "Comparecimento e abstenção de eleitores em transferência temporária (Sistema ELO, redistribution_allowed: true).",
        "required_columns": [
            "ANO_ELEICAO",
            "NR_TURNO",
            "SG_UF_DESTINO",
            "CD_MUNICIPIO_DESTINO",
            "NR_ZONA_DESTINO",
            "CD_GRAU_INSTRUCAO",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "ANO_ELEICAO",
            "NR_TURNO",
            "SG_UF_ORIGEM",
            "CD_MUNICIPIO_ORIGEM",
            "NM_MUNICIPIO_ORIGEM",
            "NR_ZONA_ORIGEM",
            "SG_UF_DESTINO",
            "CD_MUNICIPIO_DESTINO",
            "NM_MUNICIPIO_DESTINO",
            "NR_ZONA_DESTINO",
            "CD_GENERO",
            "DS_GENERO",
            "CD_ESTADO_CIVIL",
            "DS_ESTADO_CIVIL",
            "CD_FAIXA_ETARIA",
            "DS_FAIXA_ETARIA",
            "CD_GRAU_INSTRUCAO",
            "DS_GRAU_INSTRUCAO",
            "CD_TIPO_TRANSFERENCIA",
            "DS_TIPO_TRANSFERENCIA",
            "QT_APTOS_EM_TTE",
            "QT_COMPARECIMENTO_TTE",
            "QT_ABSTENCAO_TTE",
        ],
    },
    "bridge_correspondencia_esperada": {
        "phase": 1,
        "column_count": 21,
        "year_column": "AA_ELEICAO",
        "requires_sha512": True,
        "description": "Tabela ponte de correspondências esperadas de urnas eletrônicas por seção (correspesp/CESP_{turn}t_{uf}_<DDMMYYYYHHMM>.zip).",
        "required_columns": [
            "AA_ELEICAO",
            "CD_PLEITO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NR_ZONA",
            "NR_SECAO",
            "NR_URNA_ESPERADA",
            "CD_CARGA_URNA_ESPERADA",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "AA_ELEICAO",
            "CD_PLEITO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NM_MUNICIPIO",
            "NR_ZONA",
            "NR_SECAO",
            "NR_LOCAL_VOTACAO",
            "NR_URNA_ESPERADA",
            "CD_CARGA_URNA_ESPERADA",
            "CD_FLASHCARD_URNA_ESPERADA",
            "DT_CARGA_URNA_ESPERADA",
            "ST_CORRESP_ALTERADA",
            "NM_MAQUINA_GERACAO_MIDIA",
            "NR_SRI_TPM_GERACAO_MIDIA",
            "NR_SRI_INSTAL_GERACAO_MIDIA",
            "NM_MAQUINA_TRANSM_CORRESP",
            "NR_SRI_TPM_TRANSM_CORRESP",
            "NR_SRI_INSTAL_TRANSM_CORRESP",
        ],
    },
    "bridge_correspondencia_efetivada": {
        "phase": 1,
        "column_count": 36,
        "year_column": "AA_ELEICAO",
        "requires_sha512": True,
        "description": "Tabela ponte de correspondências efetivadas vs esperadas de urnas por seção (correspefet/CEFT_{turn}t_{uf}_<DDMMYYYYHHMM>.zip).",
        "required_columns": [
            "AA_ELEICAO",
            "CD_PLEITO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NR_ZONA",
            "NR_SECAO",
            "NR_URNA_ESPERADA",
            "NR_URNA_EFETIVADA",
            "CD_CARGA_1_URNA_ESPERADA",
            "CD_CARGA_1_URNA_EFETIVADA",
        ],
        "expected_columns": [
            "DT_GERACAO",
            "HH_GERACAO",
            "AA_ELEICAO",
            "DT_PLEITO",
            "CD_PLEITO",
            "SG_UF",
            "CD_MUNICIPIO",
            "NM_MUNICIPIO",
            "NR_ZONA",
            "NR_SECAO",
            "NR_LOCAL_VOTACAO",
            "SQ_CORRESPONDENCIA_ESPERADA",
            "NR_URNA_ESPERADA",
            "CD_CARGA_URNA_ESPERADA",
            "CD_CARGA_1_URNA_ESPERADA",
            "CD_CARGA_2_URNA_ESPERADA",
            "CD_FLASHCARD_URNA_ESPERADA",
            "DT_CARGA_URNA_ESPERADA",
            "DT_RECBTO_URNA_ESPERADA",
            "SQ_CORRESPONDENCIA_EFETIVADA",
            "NR_URNA_EFETIVADA",
            "CD_CARGA_URNA_EFETIVADA",
            "CD_CARGA_1_URNA_EFETIVADA",
            "CD_CARGA_2_URNA_EFETIVADA",
            "CD_FLASHCARD_URNA_EFETIVADA",
            "DT_CARGA_URNA_EFETIVADA",
            "DT_RECBTO_URNA_EFETIVADA",
            "CD_ORIGEM_VOTO",
            "DS_ORIGEM_VOTO",
            "TP_DIVERGENCIA",
            "NM_MAQUINA_GERACAO_MIDIA",
            "NR_SRI_TPM_GERACAO_MIDIA",
            "NR_SRI_INSTAL_GERACAO_MIDIA",
            "NM_MAQUINA_TRANSM_CORRESP",
            "NR_SRI_TPM_TRANSM_CORRESP",
            "NR_SRI_INSTAL_TRANSM_CORRESP",
        ],
    },
}


def resolve_dataset_dirs(
    preferred_exports_dir: str | Path | None = None,
    preferred_output_dir: str | Path | None = None,
) -> tuple[Path, Path]:
    """Detecta automaticamente o diretório de leitura (data/exports local ou /kaggle/input/...) e escrita."""
    candidates: list[Path] = []
    if preferred_exports_dir is not None:
        candidates.append(Path(preferred_exports_dir))
    env_dir = os.environ.get("TSE_EXPORTS_DIR")
    if env_dir:
        candidates.append(Path(env_dir))
    candidates.extend(
        [
            Path("data/exports"),
            Path("../../data/exports"),
            Path("/kaggle/input/tse-analysis-eleicoes-2026"),
            Path("/kaggle/input/datasets/daniellmattos/tse-analysis-eleicoes-2026"),
        ]
    )
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for mf in kaggle_input.rglob("manifest.json"):
            if mf.parent.name == "_checkpoints":
                candidates.append(mf.parent.parent)

    input_dir: Path | None = None
    for cand in candidates:
        if (cand / "_checkpoints" / "manifest.json").exists():
            input_dir = cand.resolve()
            break

    if input_dir is None:
        input_dir = Path(preferred_exports_dir or "data/exports").resolve()

    if preferred_output_dir is not None:
        output_dir = Path(preferred_output_dir).resolve()
    elif str(input_dir).startswith("/kaggle/input"):
        output_dir = Path("data/exports").resolve()
    else:
        output_dir = input_dir

    output_dir.mkdir(parents=True, exist_ok=True)
    return input_dir, output_dir

## Seção 2 — Auditoria de Completude das Partições Hive, Schemas 1:1 e Manifesto (`manifest.json`)

Valida a presença das 18 tabelas canônicas, ausência de prefixos intermediários ou chaves de partição proibidas, igualdade de `row_count` entre o manifesto e os rodapés Parquet, verificação `.sha512` onde exigida pelo TSE e conformidade estrita dos schemas de colunas e compressão `ZSTD`.

In [ ]:
def _resolve_partition_parquet_files(
    exports_dir: Path,
    table_name: str,
    election_year: int,
    turn: int,
    uf: str,
) -> list[Path]:
    part_dir = (
        exports_dir / table_name / f"election_year={election_year}" / f"turn={turn}" / f"uf={uf}"
    )
    if not part_dir.exists():
        return []
    return sorted(part_dir.glob("*.parquet"))


def audit_partitions_and_manifest(
    exports_dir: str | Path,
    manifest_path: str | Path | None = None,
    election_year: int = 2026,
    strict: bool = True,
) -> dict[str, Any]:
    """Audita completude de partições Hive, schemas 1:1, ZSTD e reconciliação com manifest.json."""
    exports_path = Path(exports_dir)
    m_path = (
        Path(manifest_path)
        if manifest_path is not None
        else (exports_path / "_checkpoints" / "manifest.json")
    )
    if not m_path.exists():
        raise FileNotFoundError(f"Manifesto de checkpoints ausente em: {m_path}")

    manifest_data = json.loads(m_path.read_text(encoding="utf-8"))
    partitions_map: dict[str, dict[str, Any]] = manifest_data.get("partitions", {})

    # 1. Verificar diretórios de tabela e chaves Hive no sistema de arquivos
    non_canonical_dirs: list[str] = []
    forbidden_partition_keys_found: list[str] = []

    if exports_path.exists():
        for child in sorted(exports_path.iterdir()):
            if not child.is_dir() or child.name.startswith("_") or child.name.startswith("."):
                continue
            if child.name not in TABLE_CATALOG_18:
                non_canonical_dirs.append(child.name)
            for subdir in child.rglob("*"):
                if subdir.is_dir() and "=" in subdir.name:
                    pkey = subdir.name.split("=", 1)[0]
                    if pkey in FORBIDDEN_PARTITION_KEYS or pkey not in ALLOWED_PARTITION_KEYS:
                        forbidden_partition_keys_found.append(f"{child.name}:{subdir.name}")

    # 2. Verificar presença das 18 tabelas no manifesto e reconciliar cada partição
    tables_in_manifest: set[str] = set()
    incomplete_manifest_entries: list[str] = []
    missing_partition_files: list[dict[str, Any]] = []
    row_count_mismatches: list[dict[str, Any]] = []
    unverified_sha512: list[str] = []
    schema_mismatches: list[dict[str, Any]] = []
    non_zstd_files: list[str] = []
    source_coverage_missing: list[str] = []
    missing_source_coverage_partitions: list[dict[str, Any]] = []
    source_coverage_row_count_mismatches: list[dict[str, Any]] = []

    table_stats: dict[str, dict[str, Any]] = {
        tbl: {
            "partition_count": 0,
            "manifest_rows": 0,
            "parquet_rows": 0,
            "compressed_bytes": 0,
            "column_count": TABLE_CATALOG_18[tbl]["column_count"],
            "sha512_all_verified": True,
        }
        for tbl in TABLE_CATALOG_18
    }

    for part_key, rec in sorted(partitions_map.items()):
        tbl = str(rec.get("table_name") or part_key.split("/", 1)[0])
        if tbl not in TABLE_CATALOG_18:
            non_canonical_dirs.append(tbl)
            continue
        tables_in_manifest.add(tbl)
        spec = TABLE_CATALOG_18[tbl]

        status = rec.get("status")
        if status != "completed":
            incomplete_manifest_entries.append(part_key)

        if spec["requires_sha512"] and rec.get("sha512_verified") is not True:
            unverified_sha512.append(part_key)
            table_stats[tbl]["sha512_all_verified"] = False

        p_year = int(rec.get("election_year", election_year))
        p_turn = int(rec.get("turn", 1))
        p_uf = str(rec.get("uf", ""))
        expected_rows = int(rec.get("parquet_row_count", rec.get("row_count", -1)))

        p_files = _resolve_partition_parquet_files(exports_path, tbl, p_year, p_turn, p_uf)
        if not p_files:
            missing_partition_files.append(
                {
                    "partition_key": part_key,
                    "table_name": tbl,
                    "turn": p_turn,
                    "uf": p_uf,
                    "expected_rows": expected_rows,
                    "actual_rows": 0,
                }
            )
            continue

        actual_rows = 0
        part_bytes = 0
        for pf_path in p_files:
            part_bytes += pf_path.stat().st_size
            pf = pq.ParquetFile(pf_path)
            actual_rows += pf.metadata.num_rows
            col_names = list(pf.schema_arrow.names)

            expected_cols = spec["expected_columns"]
            missing_cols = [c for c in expected_cols if c not in col_names]
            unexpected_cols = [c for c in col_names if c not in expected_cols]
            if len(col_names) != spec["column_count"] or missing_cols or unexpected_cols:
                schema_mismatches.append(
                    {
                        "partition_key": part_key,
                        "table_name": tbl,
                        "expected_column_count": spec["column_count"],
                        "actual_column_count": len(col_names),
                        "missing_columns": missing_cols,
                        "unexpected_columns": unexpected_cols,
                    }
                )

            for rg_idx in range(pf.metadata.num_row_groups):
                rg = pf.metadata.row_group(rg_idx)
                for col_idx in range(rg.num_columns):
                    comp = rg.column(col_idx).compression.upper()
                    if comp != "ZSTD":
                        non_zstd_files.append(f"{pf_path}:rg{rg_idx}:col{col_idx}={comp}")

        if actual_rows != expected_rows:
            row_count_mismatches.append(
                {
                    "partition_key": part_key,
                    "table_name": tbl,
                    "turn": p_turn,
                    "uf": p_uf,
                    "expected_rows": expected_rows,
                    "actual_rows": actual_rows,
                }
            )

        table_stats[tbl]["partition_count"] += 1
        table_stats[tbl]["manifest_rows"] += expected_rows
        table_stats[tbl]["parquet_rows"] += actual_rows
        table_stats[tbl]["compressed_bytes"] += part_bytes

    # A cobertura vem da leitura do CSV nacional, sem presumir que todas as UFs
    # necessariamente tenham registros no segundo turno.
    source_coverage = manifest_data.get("source_coverage", {})
    for tbl, spec in TABLE_CATALOG_18.items():
        if not spec.get("requires_source_coverage"):
            continue
        coverage_entries = source_coverage.get(tbl)
        scope = manifest_data.get("source_coverage_scope", {}).get(tbl)
        if not isinstance(coverage_entries, list) or not isinstance(scope, dict):
            source_coverage_missing.append(tbl)
            continue
        scoped_ufs = {str(uf).upper() for uf in scope.get("target_ufs", [])}
        scoped_turns = {int(turn) for turn in scope.get("turns", [])}
        if (
            int(scope.get("election_year", -1)) != int(election_year)
            or not EXPECTED_ELECTORAL_UFS.issubset(scoped_ufs)
            or not {1, 2}.issubset(scoped_turns)
        ):
            source_coverage_missing.append(f"{tbl}:incomplete_scope")
        if not scoped_ufs.issubset(EXPECTED_ELECTORAL_UFS) or not scoped_turns.issubset({1, 2}):
            source_coverage_missing.append(f"{tbl}:invalid_scope_values")
        seen_pairs: set[tuple[int, str]] = set()
        for entry in coverage_entries:
            if not isinstance(entry, dict) or "turn" not in entry or "uf" not in entry:
                source_coverage_missing.append(f"{tbl}:invalid_entry")
                continue
            turn = int(entry["turn"])
            uf = str(entry["uf"]).upper()
            pair = (turn, uf)
            if turn not in scoped_turns or uf not in scoped_ufs:
                source_coverage_missing.append(f"{tbl}:entry_outside_scope:{turn}:{uf}")
                continue
            if pair in seen_pairs:
                source_coverage_missing.append(f"{tbl}:duplicate:{turn}:{uf}")
                continue
            seen_pairs.add(pair)
            part_key = f"{tbl}/election_year={election_year}/turn={turn}/uf={uf}"
            rec = partitions_map.get(part_key)
            files = _resolve_partition_parquet_files(exports_path, tbl, election_year, turn, uf)
            if not rec or rec.get("status") != "completed" or not files:
                missing_source_coverage_partitions.append(
                    {"table_name": tbl, "turn": turn, "uf": uf, "partition_key": part_key}
                )
                continue
            if "row_count" in entry:
                source_rows = int(entry["row_count"])
                manifest_rows = int(rec.get("csv_row_count", rec.get("parquet_row_count", -1)))
                if source_rows != manifest_rows:
                    source_coverage_row_count_mismatches.append(
                        {"partition_key": part_key, "source_rows": source_rows, "manifest_rows": manifest_rows}
                    )

    missing_tables = sorted(set(TABLE_CATALOG_18.keys()) - tables_in_manifest)

    passed = not (
        missing_tables
        or non_canonical_dirs
        or forbidden_partition_keys_found
        or incomplete_manifest_entries
        or missing_partition_files
        or row_count_mismatches
        or unverified_sha512
        or schema_mismatches
        or non_zstd_files
        or source_coverage_missing
        or missing_source_coverage_partitions
        or source_coverage_row_count_mismatches
    )

    result = {
        "passed": passed,
        "total_tables_expected": len(TABLE_CATALOG_18),
        "total_tables_found": len(tables_in_manifest),
        "total_partitions_audited": len(partitions_map),
        "missing_tables": missing_tables,
        "non_canonical_dirs": sorted(set(non_canonical_dirs)),
        "forbidden_partition_keys": sorted(set(forbidden_partition_keys_found)),
        "incomplete_manifest_entries": incomplete_manifest_entries,
        "missing_partition_files": missing_partition_files,
        "row_count_mismatches": row_count_mismatches,
        "unverified_sha512": unverified_sha512,
        "schema_mismatches": schema_mismatches,
        "non_zstd_files": non_zstd_files,
        "source_coverage_missing": source_coverage_missing,
        "missing_source_coverage_partitions": missing_source_coverage_partitions,
        "source_coverage_row_count_mismatches": source_coverage_row_count_mismatches,
        "table_stats": table_stats,
    }

    if strict and not passed:
        raise ValueError(
            "Falha na auditoria física de partições/schemas/manifesto: "
            + json.dumps(
                {k: v for k, v in result.items() if k not in ("table_stats", "passed") and v},
                ensure_ascii=False,
            )
        )
    return result

## Seção 3 — Validação de Integridade Referencial e Cruzamento de Chaves via DuckDB

Executa as três consultas analíticas em DuckDB diretamente sobre os arquivos Parquet ZSTD:
1. Reconciliação de seções, votos (`QT_VOTOS`) e comparecimento (`QT_APTOS`, `QT_COMPARECIMENTO`, `QT_ABSTENCOES`) entre `fact_boletim_urna`, `fact_votacao_secao` e `fact_detalhe_votacao_secao`.
2. Cruzamento de `urna_serial` (`fact_urna_logs`) com `NR_URNA_EFETIVADA` (`fact_boletim_urna` e `bridge_correspondencia_efetivada`) e `NR_URNA_ESPERADA` (`bridge_correspondencia_esperada`), reportando cobertura explícita por UF e turno e preservando `missing_log` / `NULL` para seções sem arquivo `.jez`.
3. Consistência contábil e quantitativa de eleitores aptos entre `fact_eleitorado_perfil_secao`, `dim_local_votacao_secao` e `fact_comparecimento_abstencao` (`QT_APTOS == QT_COMPARECIMENTO + QT_ABSTENCAO`).

In [ ]:
def _open_duckdb_bounded(
    memory_limit: str = "6GB",
) -> tuple[duckdb.DuckDBPyConnection, tempfile.TemporaryDirectory[str]]:
    tmp_dir = tempfile.TemporaryDirectory(prefix="tse_duckdb_")
    con = duckdb.connect(":memory:")
    con.execute(f"SET memory_limit='{memory_limit}'")
    con.execute(f"SET temp_directory='{tmp_dir.name}'")
    return con, tmp_dir


def _uf_glob_list(
    exports_dir: Path, table_name: str, target_ufs: list[str] | None = None
) -> list[str]:
    base = exports_dir / table_name
    if not target_ufs:
        return [str(base / "**" / "*.parquet")]
    globs: list[str] = []
    for uf in sorted({u.upper() for u in target_ufs}):
        matches = list(base.glob(f"election_year=*/turn=*/uf={uf}/*.parquet"))
        if matches:
            globs.extend(str(m) for m in sorted(matches))
    return globs if globs else [str(base / "**" / "*.parquet")]


def reconcile_section_votes_and_turnout(
    exports_dir: str | Path,
    target_ufs: list[str] | None = None,
) -> dict[str, Any]:
    """Reconcilia seções, votos (QT_VOTOS) e comparecimento entre BU, votacao_secao e detalhe_votacao_secao via DuckDB."""
    root = Path(exports_dir)
    bu_globs = _uf_glob_list(root, "fact_boletim_urna", target_ufs)
    vs_globs = _uf_glob_list(root, "fact_votacao_secao", target_ufs)
    dvs_globs = _uf_glob_list(root, "fact_detalhe_votacao_secao", target_ufs)

    con, tmp_dir = _open_duckdb_bounded()
    try:
        query = """
        WITH bu AS (
            SELECT
                CAST(ANO_ELEICAO AS BIGINT) AS ano_eleicao,
                CAST(NR_TURNO AS BIGINT) AS nr_turno,
                CAST(SG_UF AS VARCHAR) AS sg_uf,
                TRIM(CAST(CD_MUNICIPIO AS VARCHAR)) AS cd_municipio,
                TRIM(CAST(NR_ZONA AS VARCHAR)) AS nr_zona,
                TRIM(CAST(NR_SECAO AS VARCHAR)) AS nr_secao,
                CAST(CD_CARGO_PERGUNTA AS BIGINT) AS cd_cargo,
                SUM(CAST(QT_VOTOS AS BIGINT)) AS bu_votes,
                MAX(CAST(QT_APTOS AS BIGINT)) AS bu_aptos,
                MAX(CAST(QT_COMPARECIMENTO AS BIGINT)) AS bu_comp,
                MAX(CAST(QT_ABSTENCOES AS BIGINT)) AS bu_abst,
                MAX(
                    CASE
                        WHEN DS_SECOES_AGREGADAS IS NOT NULL
                         AND TRIM(CAST(DS_SECOES_AGREGADAS AS VARCHAR)) NOT IN ('', '#NULO#', '-1')
                        THEN 1 ELSE 0
                    END
                ) AS has_aggregated_sections
            FROM read_parquet(?, hive_partitioning=true, union_by_name=true)
            GROUP BY 1, 2, 3, 4, 5, 6, 7
        ),
        vs AS (
            SELECT
                CAST(ANO_ELEICAO AS BIGINT) AS ano_eleicao,
                CAST(NR_TURNO AS BIGINT) AS nr_turno,
                CAST(SG_UF AS VARCHAR) AS sg_uf,
                TRIM(CAST(CD_MUNICIPIO AS VARCHAR)) AS cd_municipio,
                TRIM(CAST(NR_ZONA AS VARCHAR)) AS nr_zona,
                TRIM(CAST(NR_SECAO AS VARCHAR)) AS nr_secao,
                CAST(CD_CARGO AS BIGINT) AS cd_cargo,
                SUM(CAST(QT_VOTOS AS BIGINT)) AS vs_votes
            FROM read_parquet(?, hive_partitioning=true, union_by_name=true)
            GROUP BY 1, 2, 3, 4, 5, 6, 7
        ),
        dvs AS (
            SELECT
                CAST(ANO_ELEICAO AS BIGINT) AS ano_eleicao,
                CAST(NR_TURNO AS BIGINT) AS nr_turno,
                CAST(SG_UF AS VARCHAR) AS sg_uf,
                TRIM(CAST(CD_MUNICIPIO AS VARCHAR)) AS cd_municipio,
                TRIM(CAST(NR_ZONA AS VARCHAR)) AS nr_zona,
                TRIM(CAST(NR_SECAO AS VARCHAR)) AS nr_secao,
                CAST(CD_CARGO AS BIGINT) AS cd_cargo,
                SUM(
                    COALESCE(CAST(QT_VOTOS_NOMINAIS AS BIGINT), 0)
                    + COALESCE(CAST(QT_VOTOS_BRANCOS AS BIGINT), 0)
                    + COALESCE(CAST(QT_VOTOS_NULOS AS BIGINT), 0)
                    + COALESCE(CAST(QT_VOTOS_LEGENDA AS BIGINT), 0)
                    + COALESCE(CAST(QT_VOTOS_ANULADOS_APU_SEP AS BIGINT), 0)
                ) AS dvs_votes,
                MAX(CAST(QT_APTOS AS BIGINT)) AS dvs_aptos,
                MAX(CAST(QT_COMPARECIMENTO AS BIGINT)) AS dvs_comp,
                MAX(CAST(QT_ABSTENCOES AS BIGINT)) AS dvs_abst,
                MAX(UPPER(TRIM(CAST(ST_SECAO_ANULADA AS VARCHAR)))) AS st_secao_anulada
            FROM read_parquet(?, hive_partitioning=true, union_by_name=true)
            GROUP BY 1, 2, 3, 4, 5, 6, 7
        ),
        joined AS (
            SELECT
                COALESCE(bu.ano_eleicao, vs.ano_eleicao, dvs.ano_eleicao) AS ano_eleicao,
                COALESCE(bu.nr_turno, vs.nr_turno, dvs.nr_turno) AS nr_turno,
                COALESCE(bu.sg_uf, vs.sg_uf, dvs.sg_uf) AS sg_uf,
                COALESCE(bu.cd_municipio, vs.cd_municipio, dvs.cd_municipio) AS cd_municipio,
                COALESCE(bu.nr_zona, vs.nr_zona, dvs.nr_zona) AS nr_zona,
                COALESCE(bu.nr_secao, vs.nr_secao, dvs.nr_secao) AS nr_secao,
                COALESCE(bu.cd_cargo, vs.cd_cargo, dvs.cd_cargo) AS cd_cargo,
                bu.bu_votes,
                vs.vs_votes,
                dvs.dvs_votes,
                bu.bu_aptos,
                dvs.dvs_aptos,
                bu.bu_comp,
                dvs.dvs_comp,
                bu.bu_abst,
                dvs.dvs_abst,
                COALESCE(bu.has_aggregated_sections, 0) AS has_aggregated_sections,
                COALESCE(dvs.st_secao_anulada, 'N') AS st_secao_anulada,
                CASE
                    WHEN bu.bu_votes IS NULL OR vs.vs_votes IS NULL OR dvs.dvs_votes IS NULL THEN 'orphan_slice'
                    WHEN COALESCE(dvs.st_secao_anulada, 'N') = 'S' THEN 'annulled_section'
                    WHEN bu.bu_votes = vs.vs_votes AND vs.vs_votes = dvs.dvs_votes
                     AND bu.bu_aptos = dvs.dvs_aptos
                     AND bu.bu_comp = dvs.dvs_comp
                     AND bu.bu_abst = dvs.dvs_abst THEN 'exact_match'
                    WHEN COALESCE(bu.has_aggregated_sections, 0) = 1 THEN 'aggregated_difference'
                    ELSE 'unexplained_divergence'
                END AS reconciliation_status
            FROM bu
            FULL OUTER JOIN vs
              ON bu.ano_eleicao = vs.ano_eleicao
             AND bu.nr_turno = vs.nr_turno
             AND bu.sg_uf = vs.sg_uf
             AND bu.cd_municipio = vs.cd_municipio
             AND bu.nr_zona = vs.nr_zona
             AND bu.nr_secao = vs.nr_secao
             AND bu.cd_cargo = vs.cd_cargo
            FULL OUTER JOIN dvs
              ON COALESCE(bu.ano_eleicao, vs.ano_eleicao) = dvs.ano_eleicao
             AND COALESCE(bu.nr_turno, vs.nr_turno) = dvs.nr_turno
             AND COALESCE(bu.sg_uf, vs.sg_uf) = dvs.sg_uf
             AND COALESCE(bu.cd_municipio, vs.cd_municipio) = dvs.cd_municipio
             AND COALESCE(bu.nr_zona, vs.nr_zona) = dvs.nr_zona
             AND COALESCE(bu.nr_secao, vs.nr_secao) = dvs.nr_secao
             AND COALESCE(bu.cd_cargo, vs.cd_cargo) = dvs.cd_cargo
        )
        SELECT
            nr_turno,
            sg_uf,
            reconciliation_status,
            COUNT(*) AS slice_count,
            COALESCE(SUM(bu_votes), 0) AS sum_bu_votes,
            COALESCE(SUM(vs_votes), 0) AS sum_vs_votes,
            COALESCE(SUM(dvs_votes), 0) AS sum_dvs_votes
        FROM joined
        GROUP BY 1, 2, 3
        ORDER BY 1, 2, 3
        """
        rows = con.execute(query, [bu_globs, vs_globs, dvs_globs]).fetchall()
        cols = [
            "nr_turno",
            "sg_uf",
            "reconciliation_status",
            "slice_count",
            "sum_bu_votes",
            "sum_vs_votes",
            "sum_dvs_votes",
        ]
        summary_by_uf_turn = [dict(zip(cols, r, strict=True)) for r in rows]

        status_totals: dict[str, int] = {}
        for item in summary_by_uf_turn:
            st = str(item["reconciliation_status"])
            status_totals[st] = status_totals.get(st, 0) + int(item["slice_count"])

        return {
            "total_slices": sum(status_totals.values()),
            "exact_match_slices": status_totals.get("exact_match", 0),
            "annulled_section_slices": status_totals.get("annulled_section", 0),
            "aggregated_difference_slices": status_totals.get("aggregated_difference", 0),
            "orphan_slices": status_totals.get("orphan_slice", 0),
            "unexplained_divergence_slices": status_totals.get("unexplained_divergence", 0),
            "summary_by_uf_turn": summary_by_uf_turn,
        }
    finally:
        con.close()
        tmp_dir.cleanup()
        gc.collect()


def cross_check_urna_serial_and_log_coverage(
    exports_dir: str | Path,
    target_ufs: list[str] | None = None,
) -> dict[str, Any]:
    """Cruza urna_serial (fact_urna_logs) com NR_URNA_EFETIVADA e NR_URNA_ESPERADA preservando missing_log / NULL."""
    root = Path(exports_dir)
    bu_globs = _uf_glob_list(root, "fact_boletim_urna", target_ufs)
    ceft_globs = _uf_glob_list(root, "bridge_correspondencia_efetivada", target_ufs)
    cesp_globs = _uf_glob_list(root, "bridge_correspondencia_esperada", target_ufs)
    ul_globs = _uf_glob_list(root, "fact_urna_logs", target_ufs)

    con, tmp_dir = _open_duckdb_bounded()
    try:
        section_cte = """
        WITH bu_sec AS (
            SELECT
                CAST(ANO_ELEICAO AS BIGINT) AS election_year,
                CAST(NR_TURNO AS BIGINT) AS turn,
                CAST(SG_UF AS VARCHAR) AS uf,
                LTRIM(TRIM(CAST(CD_MUNICIPIO AS VARCHAR)), '0') AS cd_municipio_norm,
                LTRIM(TRIM(CAST(NR_ZONA AS VARCHAR)), '0') AS nr_zona_norm,
                LTRIM(TRIM(CAST(NR_SECAO AS VARCHAR)), '0') AS nr_secao_norm,
                TRIM(CAST(CD_MUNICIPIO AS VARCHAR)) AS cd_municipio,
                TRIM(CAST(NR_ZONA AS VARCHAR)) AS nr_zona,
                TRIM(CAST(NR_SECAO AS VARCHAR)) AS nr_secao,
                MAX(LTRIM(TRIM(CAST(NR_URNA_EFETIVADA AS VARCHAR)), '0')) AS bu_urna_efetivada
            FROM read_parquet(?, hive_partitioning=true, union_by_name=true)
            GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9
        ),
        ceft_sec AS (
            SELECT
                CAST(AA_ELEICAO AS BIGINT) AS election_year,
                CAST(turn AS BIGINT) AS turn,
                CAST(SG_UF AS VARCHAR) AS uf,
                LTRIM(TRIM(CAST(CD_MUNICIPIO AS VARCHAR)), '0') AS cd_municipio_norm,
                LTRIM(TRIM(CAST(NR_ZONA AS VARCHAR)), '0') AS nr_zona_norm,
                LTRIM(TRIM(CAST(NR_SECAO AS VARCHAR)), '0') AS nr_secao_norm,
                MAX(LTRIM(TRIM(CAST(NR_URNA_EFETIVADA AS VARCHAR)), '0')) AS ceft_urna_efetivada,
                MAX(LTRIM(TRIM(CAST(NR_URNA_ESPERADA AS VARCHAR)), '0')) AS ceft_urna_esperada
            FROM read_parquet(?, hive_partitioning=true, union_by_name=true)
            GROUP BY 1, 2, 3, 4, 5, 6
        ),
        cesp_sec AS (
            SELECT
                CAST(AA_ELEICAO AS BIGINT) AS election_year,
                CAST(turn AS BIGINT) AS turn,
                CAST(SG_UF AS VARCHAR) AS uf,
                LTRIM(TRIM(CAST(CD_MUNICIPIO AS VARCHAR)), '0') AS cd_municipio_norm,
                LTRIM(TRIM(CAST(NR_ZONA AS VARCHAR)), '0') AS nr_zona_norm,
                LTRIM(TRIM(CAST(NR_SECAO AS VARCHAR)), '0') AS nr_secao_norm,
                MAX(LTRIM(TRIM(CAST(NR_URNA_ESPERADA AS VARCHAR)), '0')) AS cesp_urna_esperada
            FROM read_parquet(?, hive_partitioning=true, union_by_name=true)
            GROUP BY 1, 2, 3, 4, 5, 6
        ),
        ul_sec AS (
            SELECT
                CAST(election_year AS BIGINT) AS election_year,
                CAST(turn AS BIGINT) AS turn,
                CAST(uf AS VARCHAR) AS uf,
                LTRIM(TRIM(CAST(cd_municipio AS VARCHAR)), '0') AS cd_municipio_norm,
                LTRIM(TRIM(CAST(nr_zona AS VARCHAR)), '0') AS nr_zona_norm,
                LTRIM(TRIM(CAST(nr_secao AS VARCHAR)), '0') AS nr_secao_norm,
                MAX(LTRIM(TRIM(CAST(urna_serial AS VARCHAR)), '0')) AS log_urna_serial,
                COUNT(*) AS log_event_count
            FROM read_parquet(?, hive_partitioning=true, union_by_name=true)
            GROUP BY 1, 2, 3, 4, 5, 6
        ),
        cross_sections AS (
            SELECT
                b.election_year,
                b.turn,
                b.uf,
                b.cd_municipio,
                b.nr_zona,
                b.nr_secao,
                b.bu_urna_efetivada,
                COALESCE(ce.ceft_urna_efetivada, b.bu_urna_efetivada) AS nr_urna_efetivada,
                COALESCE(ce.ceft_urna_esperada, cs.cesp_urna_esperada) AS nr_urna_esperada,
                u.log_urna_serial,
                u.log_event_count,
                CASE
                    WHEN u.log_event_count IS NULL THEN 'missing_log'
                    ELSE 'covered'
                END AS coverage_status,
                CASE
                    WHEN u.log_urna_serial IS NOT NULL
                     AND u.log_urna_serial = COALESCE(ce.ceft_urna_efetivada, b.bu_urna_efetivada)
                    THEN 1 ELSE 0
                END AS is_exact_serial_match,
                CASE
                    WHEN COALESCE(ce.ceft_urna_esperada, cs.cesp_urna_esperada) IS NOT NULL
                     AND COALESCE(ce.ceft_urna_efetivada, b.bu_urna_efetivada) IS NOT NULL
                     AND COALESCE(ce.ceft_urna_esperada, cs.cesp_urna_esperada)
                         != COALESCE(ce.ceft_urna_efetivada, b.bu_urna_efetivada)
                    THEN 1 ELSE 0
                END AS is_contingency_swap
            FROM bu_sec b
            LEFT JOIN ceft_sec ce
              ON b.election_year = ce.election_year
             AND b.turn = ce.turn
             AND b.uf = ce.uf
             AND b.cd_municipio_norm = ce.cd_municipio_norm
             AND b.nr_zona_norm = ce.nr_zona_norm
             AND b.nr_secao_norm = ce.nr_secao_norm
            LEFT JOIN cesp_sec cs
              ON b.election_year = cs.election_year
             AND b.turn = cs.turn
             AND b.uf = cs.uf
             AND b.cd_municipio_norm = cs.cd_municipio_norm
             AND b.nr_zona_norm = cs.nr_zona_norm
             AND b.nr_secao_norm = cs.nr_secao_norm
            LEFT JOIN ul_sec u
              ON b.election_year = u.election_year
             AND b.turn = u.turn
             AND b.uf = u.uf
             AND b.cd_municipio_norm = u.cd_municipio_norm
             AND b.nr_zona_norm = u.nr_zona_norm
             AND b.nr_secao_norm = u.nr_secao_norm
        )
        """
        summary_sql = (
            section_cte
            + """
        SELECT
            turn,
            uf,
            COUNT(*) AS total_bu_sections,
            SUM(CASE WHEN coverage_status = 'covered' THEN 1 ELSE 0 END) AS sections_with_logs,
            SUM(CASE WHEN coverage_status = 'missing_log' THEN 1 ELSE 0 END) AS missing_log_sections,
            ROUND(
                100.0 * SUM(CASE WHEN coverage_status = 'covered' THEN 1 ELSE 0 END)
                / NULLIF(COUNT(*), 0),
                4
            ) AS log_coverage_pct,
            SUM(is_exact_serial_match) AS exact_serial_match_count,
            SUM(is_contingency_swap) AS contingency_urna_swap_count
        FROM cross_sections
        GROUP BY 1, 2
        ORDER BY 1, 2
        """
        )
        params = [bu_globs, ceft_globs, cesp_globs, ul_globs]
        summary_rows = con.execute(summary_sql, params).fetchall()
        s_cols = [
            "turn",
            "uf",
            "total_bu_sections",
            "sections_with_logs",
            "missing_log_sections",
            "log_coverage_pct",
            "exact_serial_match_count",
            "contingency_urna_swap_count",
        ]
        coverage_by_uf_turn = [dict(zip(s_cols, r, strict=True)) for r in summary_rows]

        sample_sql = (
            section_cte
            + """
        (
            SELECT turn, uf, cd_municipio, nr_zona, nr_secao,
                   nr_urna_efetivada, nr_urna_esperada, log_urna_serial,
                   coverage_status, log_event_count
            FROM cross_sections
            WHERE coverage_status = 'covered'
            ORDER BY turn, uf, cd_municipio, nr_zona, nr_secao
            LIMIT 10
        )
        UNION ALL
        (
            SELECT turn, uf, cd_municipio, nr_zona, nr_secao,
                   nr_urna_efetivada, nr_urna_esperada, log_urna_serial,
                   coverage_status, log_event_count
            FROM cross_sections
            WHERE coverage_status = 'missing_log'
            ORDER BY turn, uf, cd_municipio, nr_zona, nr_secao
            LIMIT 10
        )
        """
        )
        sample_rows = con.execute(sample_sql, params).fetchall()
        sec_cols = [
            "turn",
            "uf",
            "cd_municipio",
            "nr_zona",
            "nr_secao",
            "nr_urna_efetivada",
            "nr_urna_esperada",
            "log_urna_serial",
            "coverage_status",
            "log_event_count",
        ]
        sample_sections = [dict(zip(sec_cols, r, strict=True)) for r in sample_rows]

        total_bu = sum(int(r["total_bu_sections"]) for r in coverage_by_uf_turn)
        total_cov = sum(int(r["sections_with_logs"]) for r in coverage_by_uf_turn)
        total_miss = sum(int(r["missing_log_sections"]) for r in coverage_by_uf_turn)
        total_match = sum(int(r["exact_serial_match_count"]) for r in coverage_by_uf_turn)
        total_swap = sum(int(r["contingency_urna_swap_count"]) for r in coverage_by_uf_turn)

        return {
            "total_bu_sections": total_bu,
            "sections_with_logs": total_cov,
            "missing_log_sections": total_miss,
            "overall_log_coverage_pct": round(100.0 * total_cov / total_bu, 4) if total_bu else 0.0,
            "exact_serial_match_count": total_match,
            "contingency_urna_swap_count": total_swap,
            "coverage_by_uf_turn": coverage_by_uf_turn,
            "sample_sections": sample_sections,
        }
    finally:
        con.close()
        tmp_dir.cleanup()
        gc.collect()


def audit_electorate_and_turnout_consistency(
    exports_dir: str | Path,
    target_ufs: list[str] | None = None,
) -> dict[str, Any]:
    """Valida consistência de eleitores aptos e identidade contábil QT_APTOS == QT_COMPARECIMENTO + QT_ABSTENCAO."""
    root = Path(exports_dir)
    eps_globs = _uf_glob_list(root, "fact_eleitorado_perfil_secao", target_ufs)
    lvs_globs = _uf_glob_list(root, "dim_local_votacao_secao", target_ufs)
    ca_globs = _uf_glob_list(root, "fact_comparecimento_abstencao", target_ufs)
    tte_globs = _uf_glob_list(root, "fact_transferencia_temporaria_secao", target_ufs)

    con, tmp_dir = _open_duckdb_bounded()
    try:
        inv_row = con.execute(
            """
            SELECT
                COUNT(*) AS total_rows,
                SUM(
                    CASE
                        WHEN CAST(QT_APTOS AS BIGINT)
                             != (CAST(QT_COMPARECIMENTO AS BIGINT) + CAST(QT_ABSTENCAO AS BIGINT))
                        THEN 1 ELSE 0
                    END
                ) AS accounting_violations
            FROM read_parquet(?, hive_partitioning=true, union_by_name=true)
            """,
            [ca_globs],
        ).fetchone()
        ca_total_rows = int(inv_row[0] or 0)
        accounting_violations = int(inv_row[1] or 0)

        align_sql = """
        WITH eps AS (
            SELECT
                CAST(SG_UF AS VARCHAR) AS sg_uf,
                SUM(CAST(QT_ELEITORES AS BIGINT)) AS eps_eleitores,
                COUNT(DISTINCT CONCAT(CD_MUNICIPIO, ':', NR_ZONA, ':', NR_SECAO)) AS eps_sections
            FROM read_parquet(?, hive_partitioning=true, union_by_name=true)
            GROUP BY 1
        ),
        lvs AS (
            SELECT
                CAST(SG_UF AS VARCHAR) AS sg_uf,
                SUM(CAST(QT_ELEITOR_SECAO AS BIGINT)) AS lvs_eleitores_t1,
                COUNT(DISTINCT CONCAT(CD_MUNICIPIO, ':', NR_ZONA, ':', NR_SECAO)) AS lvs_sections_t1,
                SUM(
                    CASE
                        WHEN NR_SECAO_PRINCIPAL IS NOT NULL
                         AND TRIM(CAST(NR_SECAO_PRINCIPAL AS VARCHAR)) NOT IN ('', '-1', '#NULO#')
                         AND TRIM(CAST(NR_SECAO_PRINCIPAL AS VARCHAR)) != TRIM(CAST(NR_SECAO AS VARCHAR))
                        THEN 1 ELSE 0
                    END
                ) AS aggregated_section_records
            FROM read_parquet(?, hive_partitioning=true, union_by_name=true)
            WHERE CAST(NR_TURNO AS BIGINT) = 1
            GROUP BY 1
        ),
        ca AS (
            SELECT
                CAST(SG_UF AS VARCHAR) AS sg_uf,
                SUM(CASE WHEN CAST(NR_TURNO AS BIGINT) = 1 THEN CAST(QT_APTOS AS BIGINT) ELSE 0 END) AS ca_aptos_t1,
                SUM(CASE WHEN CAST(NR_TURNO AS BIGINT) = 1 THEN CAST(QT_COMPARECIMENTO AS BIGINT) ELSE 0 END) AS ca_comp_t1,
                SUM(CASE WHEN CAST(NR_TURNO AS BIGINT) = 1 THEN CAST(QT_ABSTENCAO AS BIGINT) ELSE 0 END) AS ca_abst_t1
            FROM read_parquet(?, hive_partitioning=true, union_by_name=true)
            GROUP BY 1
        ),
        tte AS (
            SELECT
                CAST(SG_UF_DESTINO AS VARCHAR) AS sg_uf,
                SUM(CASE WHEN CAST(NR_TURNO AS BIGINT) = 1 THEN CAST(QT_ELEITOR AS BIGINT) ELSE 0 END) AS tte_dest_t1
            FROM read_parquet(?, hive_partitioning=true, union_by_name=true)
            GROUP BY 1
        )
        SELECT
            COALESCE(eps.sg_uf, lvs.sg_uf, ca.sg_uf) AS sg_uf,
            COALESCE(eps.eps_eleitores, 0) AS eps_eleitores,
            COALESCE(lvs.lvs_eleitores_t1, 0) AS lvs_eleitores_t1,
            COALESCE(ca.ca_aptos_t1, 0) AS ca_aptos_t1,
            COALESCE(ca.ca_comp_t1, 0) AS ca_comp_t1,
            COALESCE(ca.ca_abst_t1, 0) AS ca_abst_t1,
            COALESCE(lvs.aggregated_section_records, 0) AS aggregated_section_records,
            COALESCE(tte.tte_dest_t1, 0) AS tte_dest_t1
        FROM eps
        FULL OUTER JOIN lvs ON eps.sg_uf = lvs.sg_uf
        FULL OUTER JOIN ca ON COALESCE(eps.sg_uf, lvs.sg_uf) = ca.sg_uf
        LEFT JOIN tte ON COALESCE(eps.sg_uf, lvs.sg_uf, ca.sg_uf) = tte.sg_uf
        ORDER BY 1
        """
        align_rows = con.execute(align_sql, [eps_globs, lvs_globs, ca_globs, tte_globs]).fetchall()
        a_cols = [
            "sg_uf",
            "eps_eleitores",
            "lvs_eleitores_t1",
            "ca_aptos_t1",
            "ca_comp_t1",
            "ca_abst_t1",
            "aggregated_section_records",
            "tte_dest_t1",
        ]
        by_uf = [dict(zip(a_cols, r, strict=True)) for r in align_rows]

        return {
            "comparecimento_total_rows": ca_total_rows,
            "accounting_identity_violations": accounting_violations,
            "by_uf": by_uf,
        }
    finally:
        con.close()
        tmp_dir.cleanup()
        gc.collect()

## Seção 4 — Auditoria Final de Privacidade, Orçamento de Volume ZSTD e Prontidão do Pacote para o Kaggle

1. Confirma a ausência física da coluna `mac_code` em todas as partições de `fact_urna_logs` e verifica via DuckDB (`regexp_matches(message_text, '\b[0-9]{5,}\b')`) que existem zero números de 5+ dígitos desmascarados.
2. Calcula o volume comprimido ZSTD por tabela e total acumulado.
3. Gera `dataset-metadata.json` e `integrity_audit_report.json` como evidências auditáveis para conferência manual pelo responsável antes da publicação no Kaggle.

In [ ]:
def audit_urna_logs_privacy(exports_dir: str | Path) -> dict[str, Any]:
    """Audita ausência física de mac_code e 100% de mascaramento <ID_REDACTED> para números de 5+ dígitos."""
    root = Path(exports_dir)
    ul_dir = root / "fact_urna_logs"
    p_files = sorted(ul_dir.rglob("*.parquet")) if ul_dir.exists() else []
    if not p_files:
        raise PrivacyAuditError(
            f"Nenhuma partição Parquet encontrada para fact_urna_logs em {ul_dir}"
        )

    files_with_mac_code: list[str] = []
    for pf_path in p_files:
        pf = pq.ParquetFile(pf_path)
        if "mac_code" in pf.schema_arrow.names:
            files_with_mac_code.append(str(pf_path))

    if files_with_mac_code:
        raise PrivacyAuditError(
            f"Coluna proibida 'mac_code' encontrada em {len(files_with_mac_code)} arquivo(s): {files_with_mac_code[:3]}"
        )

    con, tmp_dir = _open_duckdb_bounded()
    try:
        glob_str = [str(p) for p in p_files]
        row = con.execute(
            """
            SELECT
                COUNT(*) AS total_log_rows,
                SUM(CASE WHEN regexp_matches(CAST(message_text AS VARCHAR), '\\b[0-9]{5,}\\b') THEN 1 ELSE 0 END) AS unmasked_5plus_digit_rows,
                SUM(CASE WHEN CAST(message_text AS VARCHAR) LIKE '%<ID_REDACTED>%' THEN 1 ELSE 0 END) AS redacted_token_rows
            FROM read_parquet(?, hive_partitioning=true, union_by_name=true)
            """,
            [glob_str],
        ).fetchone()
        total_log_rows = int(row[0] or 0)
        unmasked_rows = int(row[1] or 0)
        redacted_rows = int(row[2] or 0)

        if unmasked_rows > 0:
            raise PrivacyAuditError(
                f"Violação de privacidade: {unmasked_rows} linha(s) em fact_urna_logs.message_text contêm sequências numéricas de 5+ dígitos sem <ID_REDACTED>."
            )

        return {
            "passed": True,
            "has_mac_code_column": False,
            "parquet_files_checked": len(p_files),
            "total_log_rows": total_log_rows,
            "unmasked_5plus_digit_rows": unmasked_rows,
            "redacted_token_rows": redacted_rows,
        }
    finally:
        con.close()
        tmp_dir.cleanup()
        gc.collect()


def _arrow_type_to_kaggle_type(arrow_type_str: str) -> str:
    t = arrow_type_str.lower()
    if any(k in t for k in ("int", "float", "double", "decimal")):
        return "numeric"
    if "bool" in t:
        return "boolean"
    if any(k in t for k in ("date", "time", "timestamp")):
        return "datetime"
    return "string"


def compute_zstd_storage_budget_and_generate_metadata(
    exports_dir: str | Path,
    output_dir: str | Path | None = None,
    dataset_id: str = "daniellmattos/tse-analysis-eleicoes-2026",
    max_budget_gb: float = 100.0,
) -> dict[str, Any]:
    """Mede o orçamento ZSTD das 18 tabelas e gera data/exports/dataset-metadata.json completo."""
    root = Path(exports_dir)
    out_root = Path(output_dir) if output_dir is not None else root
    out_root.mkdir(parents=True, exist_ok=True)

    table_sizes: dict[str, dict[str, Any]] = {}
    resources: list[dict[str, Any]] = []
    total_bytes = 0

    for tbl_name, spec in TABLE_CATALOG_18.items():
        tbl_dir = root / tbl_name
        p_files = sorted(tbl_dir.rglob("*.parquet")) if tbl_dir.exists() else []
        tbl_bytes = sum(p.stat().st_size for p in p_files)
        total_bytes += tbl_bytes

        fields: list[dict[str, str]] = []
        if p_files:
            pf = pq.ParquetFile(p_files[0])
            for field in pf.schema_arrow:
                fields.append(
                    {
                        "name": field.name,
                        "type": _arrow_type_to_kaggle_type(str(field.type)),
                        "description": f"Coluna nativa 1:1 {field.name} da tabela {tbl_name} (TSE 2026).",
                    }
                )
        else:
            for col_name in spec["expected_columns"]:
                fields.append(
                    {
                        "name": col_name,
                        "type": "string",
                        "description": f"Coluna nativa 1:1 {col_name} da tabela {tbl_name} (TSE 2026).",
                    }
                )

        table_sizes[tbl_name] = {
            "parquet_files": len(p_files),
            "compressed_bytes": tbl_bytes,
            "compressed_gb": round(tbl_bytes / (1024**3), 6),
            "column_count": len(fields),
        }

        resources.append(
            {
                "path": tbl_name,
                "description": spec["description"],
                "schema": {"fields": fields},
            }
        )

    total_gb = round(total_bytes / (1024**3), 6)
    within_budget = total_gb <= max_budget_gb
    if not within_budget:
        raise ValueError(
            f"Volume total ZSTD ({total_gb} GB) excede o orçamento máximo ({max_budget_gb} GB)."
        )

    urls_md = "\n".join(f"- `{u}`" for u in VERIFIED_TSE_2026_URLS)
    description_md = (
        "# Eleições Gerais 2026 (TSE) — Dataset Analítico 1:1 em Parquet ZSTD\n\n"
        "Espelhamento analítico 1:1 em Parquet ZSTD (nível 5) particionado em formato Hive "
        "(`election_year=2026/turn={1,2}/uf={UF}`) contendo as **18 tabelas** oficiais de "
        "Logs da Urna Eletrônica (`fact_urna_logs`), Boletim de Urna (`fact_boletim_urna`), "
        "Resultados por Seção e Município-Zona, Eleitorado, Comparecimento/Abstenção e "
        "Tabelas Ponte de Correspondências Esperadas e Efetivadas.\n\n"
        "## Governança, Privacidade e Incorporação Pública 1:1\n"
        "- `fact_urna_logs`: coluna `mac_code` omitida na origem e sequências numéricas de 5+ dígitos "
        "mascaradas como `<ID_REDACTED>` em `message_text`.\n"
        "- `fact_comparecimento_abstencao`, `fact_comparecimento_abstencao_pcd` e "
        "`fact_comparecimento_abstencao_tte` (Sistema ELO / TSE): incorporados integralmente ao dataset "
        "público no schema 1:1 (`redistribution_allowed: true`), tratando a ausência de `license_id` no "
        "portal CKAN como omissão formal de metadado em dados públicos abertos do TSE.\n\n"
        "## Fontes Primárias Oficiais Verificadas (TSE 2026)\n"
        f"{urls_md}\n"
    )

    metadata_payload = {
        "id": dataset_id,
        "title": "TSE Eleicoes 2026 Analytics 1-to-1 Parquet ZSTD",
        "subtitle": "Espelhamento 1:1 das 18 tabelas oficiais do TSE 2026 em Parquet ZSTD",
        "licenses": [{"name": "CC-BY-4.0"}],
        "keywords": [
            "brazil",
            "elections",
            "government",
            "politics",
            "parquet",
        ],
        "description": description_md,
        "resources": resources,
    }

    metadata_path = out_root / "dataset-metadata.json"
    metadata_path.write_text(
        json.dumps(metadata_payload, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )

    return {
        "within_budget": within_budget,
        "max_budget_gb": max_budget_gb,
        "total_compressed_bytes": total_bytes,
        "total_compressed_gb": total_gb,
        "metadata_path": str(metadata_path),
        "table_sizes": table_sizes,
    }


def verify_kaggle_release_package_readiness(
    exports_dir: str | Path,
    metadata_dir: str | Path | None = None,
) -> dict[str, Any]:
    """Verifica se dataset-metadata.json + Parquets ZSTD das 18 tabelas estão prontos para publicação manual."""
    root = Path(exports_dir)
    meta_root = Path(metadata_dir) if metadata_dir is not None else root
    meta_path = meta_root / "dataset-metadata.json"
    if not meta_path.exists():
        raise FileNotFoundError(f"dataset-metadata.json ausente em: {meta_path}")

    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    title = str(meta.get("title", ""))
    subtitle = str(meta.get("subtitle", ""))
    description = str(meta.get("description", ""))
    licenses = meta.get("licenses", [])
    resources = meta.get("resources", [])

    if not (6 <= len(title) <= 50):
        raise ValueError(
            f"title fora do intervalo permitido pelo Kaggle (6..50 chars): len={len(title)}"
        )
    if not (20 <= len(subtitle) <= 80):
        raise ValueError(
            f"subtitle fora do intervalo permitido pelo Kaggle (20..80 chars): len={len(subtitle)}"
        )
    if not licenses or licenses[0].get("name") != "CC-BY-4.0":
        raise ValueError("Licença CC-BY-4.0 obrigatória ausente em dataset-metadata.json")

    missing_urls = [u for u in VERIFIED_TSE_2026_URLS if u not in description]
    if missing_urls:
        raise ValueError(f"URLs oficiais de 2026 ausentes da descrição do dataset: {missing_urls}")
    if "redistribution_allowed: true" not in description:
        raise ValueError(
            "Nota de incorporação pública 1:1 (redistribution_allowed: true) ausente da descrição"
        )

    res_by_path = {str(r.get("path")): r for r in resources}
    missing_resources = sorted(set(TABLE_CATALOG_18.keys()) - set(res_by_path.keys()))
    if missing_resources:
        raise ValueError(
            f"Tabelas ausentes de resources em dataset-metadata.json: {missing_resources}"
        )

    for tbl_name, spec in TABLE_CATALOG_18.items():
        fields = res_by_path[tbl_name].get("schema", {}).get("fields", [])
        if len(fields) != spec["column_count"]:
            raise ValueError(
                f"Contagem de colunas divergente em dataset-metadata.json para {tbl_name}: "
                f"{len(fields)} != {spec['column_count']}"
            )
        p_files = list((root / tbl_name).rglob("*.parquet"))
        if not p_files:
            raise FileNotFoundError(
                f"Nenhum arquivo Parquet encontrado para {tbl_name} em {root / tbl_name}"
            )

    return {
        "ready_for_manual_publish": True,
        "dataset_id": meta.get("id"),
        "title_length": len(title),
        "subtitle_length": len(subtitle),
        "resources_count": len(resources),
        "verified_urls_count": len(VERIFIED_TSE_2026_URLS),
        "metadata_path": str(meta_path),
    }


def run_full_phase3_integrity_audit(
    exports_dir: str | Path | None = None,
    output_dir: str | Path | None = None,
    target_ufs: list[str] | None = None,
) -> dict[str, Any]:
    """Executa a auditoria ponta a ponta da Fase 3 e grava integrity_audit_report.json e dataset-metadata.json."""
    in_dir, out_dir = resolve_dataset_dirs(exports_dir, output_dir)
    print(f"[Fase 3] Input exports_dir : {in_dir}")
    print(f"[Fase 3] Output evidences  : {out_dir}")

    partition_audit = audit_partitions_and_manifest(in_dir, strict=True)
    print(
        f"[1/5] Partições & Manifesto OK: {partition_audit['total_tables_found']}/18 tabelas | "
        f"{partition_audit['total_partitions_audited']} partições verificadas em ZSTD."
    )

    vote_reconciliation = reconcile_section_votes_and_turnout(in_dir, target_ufs=target_ufs)
    print(
        f"[2/5] Reconciliação BU vs Votação Seção vs Detalhe Seção OK: "
        f"exact_match={vote_reconciliation['exact_match_slices']} | "
        f"annulled={vote_reconciliation['annulled_section_slices']} | "
        f"aggregated={vote_reconciliation['aggregated_difference_slices']} | "
        f"unexplained={vote_reconciliation['unexplained_divergence_slices']}"
    )

    urna_serial_coverage = cross_check_urna_serial_and_log_coverage(in_dir, target_ufs=target_ufs)
    print(
        f"[3/5] Cobertura de Logs (urna_serial vs NR_URNA_EFETIVADA/ESPERADA) OK: "
        f"seções BU={urna_serial_coverage['total_bu_sections']} | "
        f"com log={urna_serial_coverage['sections_with_logs']} | "
        f"missing_log (NULL)={urna_serial_coverage['missing_log_sections']} | "
        f"cobertura={urna_serial_coverage['overall_log_coverage_pct']}%"
    )

    electorate_consistency = audit_electorate_and_turnout_consistency(in_dir, target_ufs=target_ufs)
    print(
        f"[4/5] Consistência Eleitorado & Comparecimento OK: "
        f"linhas comparecimento={electorate_consistency['comparecimento_total_rows']} | "
        f"violações contábeis (QT_APTOS != COMP + ABST)={electorate_consistency['accounting_identity_violations']}"
    )

    privacy_audit = audit_urna_logs_privacy(in_dir)
    budget_and_meta = compute_zstd_storage_budget_and_generate_metadata(in_dir, output_dir=out_dir)
    readiness = verify_kaggle_release_package_readiness(in_dir, metadata_dir=out_dir)
    print(
        f"[5/5] Privacidade & Orçamento ZSTD OK: mac_code={privacy_audit['has_mac_code_column']} | "
        f"5+ dígitos desmascarados={privacy_audit['unmasked_5plus_digit_rows']} | "
        f"<ID_REDACTED>={privacy_audit['redacted_token_rows']} | "
        f"volume total={budget_and_meta['total_compressed_gb']} GB | "
        f"ready_for_manual_publish={readiness['ready_for_manual_publish']}"
    )

    report = {
        "generated_at_utc": datetime.now(UTC).isoformat(),
        "input_exports_dir": str(in_dir),
        "output_evidences_dir": str(out_dir),
        "target_ufs_filter": target_ufs,
        "partition_and_manifest_audit": partition_audit,
        "section_vote_turnout_reconciliation": vote_reconciliation,
        "urna_serial_and_log_coverage": urna_serial_coverage,
        "electorate_and_turnout_consistency": electorate_consistency,
        "privacy_audit": privacy_audit,
        "storage_budget_and_metadata": budget_and_meta,
        "release_package_readiness": readiness,
    }

    report_path = out_dir / "integrity_audit_report.json"
    report_path.write_text(
        json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )
    print(f"[Fase 3] Relatório completo gravado em: {report_path}")
    return report

## Atualização manual do dataset no Kaggle

Depois de executar novamente as fases de ingestão e a auditoria, publique a versão atualizada a partir da raiz do repositório. O diretório `data/exports` deve conter o `dataset-metadata.json` e os arquivos Parquet finais.

```bash
kaggle datasets version \
  -p data/exports \
  -m "Atualiza dataset eleitoral 2026 após auditoria de integridade" \
  --dir-mode zip
```

O comando cria uma nova versão do dataset `daniellmattos/tse-analysis-eleicoes-2026`. Ele pressupõe que a CLI do Kaggle esteja instalada e autenticada (`~/.kaggle/kaggle.json` ou token configurado no ambiente).

In [ ]:
if __name__ == "__main__":
    # Quando executado no Kaggle Kernels ou diretamente, audita 100% das 18 tabelas nas 28 UFs e gera as evidências.
    env_ufs = os.environ.get("TSE_AUDIT_UFS")
    audit_ufs = [u.strip().upper() for u in env_ufs.split(",") if u.strip()] if env_ufs else None
    audit_report = run_full_phase3_integrity_audit(target_ufs=audit_ufs)